2day all bases --- penalty uncovere: 50000

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import gurobipy as gp
from gurobipy import GRB
from collections import defaultdict
import ast
from IPython.display import display # Jupyter-specific import for nice tables

In [2]:
print("=== STARTING MEMORY-OPTIMIZED ROSTERING LAYER ===")
print("Loading datasets...")

# 1. The NEW SCIP optimal pairings file
pairings_df = pd.read_csv(r"F:\Sushreeta MSc.DDS\2nd Sem\Analytics Project\4th Round Work\Try Rostering\pairings_ALL_2025-05-01_2025-05-02_20260718_2244.csv")  # Change path as needed

# 2. The four supporting operational datasets
flights_df = pd.read_csv(r"F:\Sushreeta MSc.DDS\2nd Sem\Analytics Project\4th Round Work\Copy_project_data\flight_schedule.csv")
home_bases_df = pd.read_csv(r"F:\Sushreeta MSc.DDS\2nd Sem\Analytics Project\4th Round Work\Copy_project_data\home_bases.csv")
off_claims_df = pd.read_csv(r"F:\Sushreeta MSc.DDS\2nd Sem\Analytics Project\4th Round Work\Copy_project_data\off_claims_202505.csv", sep=";")
off_requests_df = pd.read_csv(r"F:\Sushreeta MSc.DDS\2nd Sem\Analytics Project\4th Round Work\Copy_project_data\off_requests_202505.csv", sep=";")

# Clean European CSV headers
off_claims_df.columns = off_claims_df.columns.str.strip()
off_requests_df.columns = off_requests_df.columns.str.strip()

# Standardize the pairing ID column for the engine
pairings_df['pairing_id'] = pairings_df.get('pairing_id', pairings_df.index).astype(str)
pairings_df.set_index('pairing_id', drop=False, inplace=True)

print("✅ Data successfully loaded!")
display(pairings_df.head(1))

=== STARTING MEMORY-OPTIMIZED ROSTERING LAYER ===
Loading datasets...
✅ Data successfully loaded!


,pairing_id,base,n_legs,cost_eur,start_min,end_min,route,leg_ids
pairing_id,,,,,,,,
1,1,HAM,4,1355.0,410,2260,HAM->FUE FUE->HAM HAM->FAO FAO->HAM,3505309528|3505309529|3505310249|3505310250


In [3]:
print("Reconstructing missing features and timeline mappings...")

# 1. Safely parse the flight leg lists
def parse_legs(val):
    if pd.isna(val):
        return []
    if isinstance(val, str):
        if val.startswith('[') and val.endswith(']'):
            try:
                return ast.literal_eval(val)
            except:
                pass
        if '->' in val:
            return [int(leg.strip()) for leg in val.split('->') if leg.strip().isdigit()]
        return [int(leg.strip()) for leg in val.replace(',', ' ').split() if leg.strip().isdigit()]
    return val

pairings_df['covered_legs'] = pairings_df['leg_ids'].apply(parse_legs)

# 2. Map 'home_base'
leg_to_base_map = dict(zip(flights_df['LEG_ID'], flights_df['DEPARTURE_AIRPORT']))
if 'base' in pairings_df.columns:
    pairings_df['home_base'] = pairings_df['base']
else:
    pairings_df['home_base'] = pairings_df['covered_legs'].apply(
        lambda legs: leg_to_base_map.get(legs[0], 'DUS') if isinstance(legs, list) and len(legs) > 0 else 'DUS'
    )

# 3. Calculate 'flying_hours' dynamically 
dep_time = pd.to_datetime(flights_df['SCHEDULED_DEPARTURE_TIME'])
arr_time = pd.to_datetime(flights_df['SCHEDULED_ARRIVAL_TIME'])
flights_df['calc_hours'] = (arr_time - dep_time).dt.total_seconds() / 3600.0
leg_hours_map = dict(zip(flights_df['LEG_ID'], flights_df['calc_hours']))

pairings_df['flying_hours'] = pairings_df['covered_legs'].apply(
    lambda legs: sum(leg_hours_map.get(leg, 0.0) for leg in legs) if isinstance(legs, list) else 0.0
)

# 4. Map the exact calendar execution days
flights_df['REAL_DOM'] = pd.to_datetime(flights_df['SCHEDULED_DEPARTURE_TIME']).dt.day
leg_to_day_map = dict(zip(flights_df['LEG_ID'], flights_df['REAL_DOM']))

start_days = []
end_days = []
for idx, row in pairings_df.iterrows():
    legs = row['covered_legs']
    first_leg_day = leg_to_day_map.get(legs[0], 1) if isinstance(legs, list) and legs else 1
    last_leg_day = leg_to_day_map.get(legs[-1], first_leg_day) if isinstance(legs, list) and legs else 1
    
    start_days.append(first_leg_day)
    end_days.append(last_leg_day)

pairings_df['start_day'] = start_days
pairings_df['end_day'] = end_days

# 5. Calculate 'num_days' for fairness scaling
pairings_df['num_days'] = (pairings_df['end_day'] - pairings_df['start_day'] + 1).clip(lower=1)

# 6. Map 'total_cost'
if 'cost_eur' in pairings_df.columns:
    pairings_df['total_cost'] = pairings_df['cost_eur']
else:
    pairings_df['total_cost'] = pairings_df['flying_hours'] * 100.0

print("✅ Features reconstructed successfully!")

Reconstructing missing features and timeline mappings...
✅ Features reconstructed successfully!


In [4]:
# ==========================================
# 3. HIGH-SPEED HASH MAP INDEXING
# ==========================================
print("Building pre-indexed hash maps for timeline overlap tracking...")
pairings_by_day = defaultdict(list)
pairings_by_base = defaultdict(list)

for idx, row in pairings_df.iterrows():
    p_id = row['pairing_id']
    base = row['home_base']
    s_day = int(row['start_day'])
    e_day = int(row['end_day'])
    
    pairings_by_base[base].append(p_id)
    # Register pairing eligibility across its entire calendar span
    for d in range(s_day, e_day + 1):
        pairings_by_day[d].append(p_id)

# Parse pilot off-requests and hard leaves (KUR, U, SU)
hard_leave_codes = {'KUR', 'U', 'SU'}
captains_list = home_bases_df['captain_id'].unique().tolist()

active_days_range = sorted(list(pairings_by_day.keys()))
if not active_days_range:
    active_days_range = list(range(1, 32))

banned_captain_days = defaultdict(set)

def extract_day_int(date_str):
    try:
        return datetime.strptime(date_str.split()[0], "%d.%m.%y").day
    except:
        return 1

if 'Begin Date' in off_requests_df.columns:
    for _, row in off_requests_df.iterrows():
        c_id = row['captain_id']
        if c_id in captains_list:
            start_d = extract_day_int(row['Begin Date'])
            end_d = extract_day_int(row['End Date'])
            if str(row['Code']).strip() in hard_leave_codes:
                for d in range(start_d, end_d + 1):
                    banned_captain_days[c_id].add(d)

# ==========================================
# 4. MEMORY-OPTIMIZED GUROBI MODEL COMPILATION
# ==========================================
print("\nInitializing Gurobi Engine...")
model = gp.Model("Eurowings_Rostering_Fixed")

# Memory optimization parameters
model.setParam('Method', 2)       # Force Barrier Method to limit peak memory usage
model.setParam('NodefileStart', 0.5) # Offload to disk if RAM exceeds 500MB

# CRITICAL RAM TRICK: Prune combinations before creating decision variables
valid_y_pairs = []
for _, cap_row in home_bases_df.iterrows():
    c = cap_row['captain_id']
    c_base = cap_row['home_base']
    for p in pairings_by_base[c_base]:
        valid_y_pairs.append((c, p))

print(f"Instantiating {len(valid_y_pairs)} base-compliant decision variables...")
y = model.addVars(valid_y_pairs, vtype=GRB.BINARY, name="y")
dev = model.addVars(captains_list, vtype=GRB.CONTINUOUS, lb=0.0, name="dev")

# MUST instantiate slack variables BEFORE objective definition
print("Instantiating uncovered pairing slack variables...")
uncovered = model.addVars(pairings_df['pairing_id'], vtype=GRB.BINARY, name="uncovered")

# --- Objective Definition ---
penalty_fairness_weight = 500.0
PENALTY_UNCOVERED = 50000.0  # Heavy penalty for unassigned pairings

operational_cost_expr = gp.quicksum(y[c, p] * pairings_df.loc[p, 'total_cost'] for (c, p) in valid_y_pairs)
fairness_imbalance_expr = gp.quicksum(dev[c] * penalty_fairness_weight for c in captains_list)
uncovered_penalty_expr = gp.quicksum(uncovered[p] * PENALTY_UNCOVERED for p in pairings_df['pairing_id'])

# Unified Objective
model.setObjective(operational_cost_expr + fairness_imbalance_expr + uncovered_penalty_expr, GRB.MINIMIZE)
print("✅ Multi-objective function successfully compiled with slack penalties!")

# ==========================================
# 5. CONSTRAINT FORMULATIONS
# ==========================================
# Constraint 1: Set Partitioning Coverage with Slack Variable
print("Formulating flexible pairing coverage matrices...")
for p in pairings_df['pairing_id']:
    p_base = pairings_df.loc[p, 'home_base']
    valid_caps_for_p = home_bases_df[home_bases_df['home_base'] == p_base]['captain_id'].tolist()
    
    model.addConstr(
        gp.quicksum(y[c, p] for c in valid_caps_for_p if (c, p) in y) + uncovered[p] == 1, 
        name=f"Cover_P_{p}"
    )

# Constraint 2: Timeline Overlap & Leave Blocking
print("Formulating calendar pacing constraints...")
for c in captains_list:
    c_base = home_bases_df[home_bases_df['captain_id'] == c]['home_base'].values[0]
    base_pairings = set(pairings_by_base[c_base])
    
    for d in active_days_range:
        pairings_on_day = [p for p in pairings_by_day[d] if p in base_pairings]
        
        if not pairings_on_day:
            continue
            
        if d in banned_captain_days[c]:
            for p in pairings_on_day:
                model.addConstr(y[c, p] == 0, name=f"Leave_Block_{c}_{d}_{p}")
        else:
            model.addConstr(gp.quicksum(y[c, p] for p in pairings_on_day) <= 1, name=f"No_Overlap_{c}_{d}")

# Constraint 3: EASA Cumulative Flight Hours Limit
print("Formulating EASA fatigue limit bounds...")
for c in captains_list:
    c_base = home_bases_df[home_bases_df['captain_id'] == c]['home_base'].values[0]
    caps_pairings = pairings_by_base[c_base]
    model.addConstr(
        gp.quicksum(y[c, p] * pairings_df.loc[p, 'flying_hours'] for p in caps_pairings) <= 60.0,
        name=f"EASA_Cap_{c}"
    )

# Constraint 4: Schedule Fairness
print("Formulating absolute deviation fairness bounds...")
for c in captains_list:
    matching_claim = off_claims_df[off_claims_df['captain_id'] == c]
    # Because of the .str.strip() fix above, 'Count' is now safe to use!
    target_monthly_claim = matching_claim['Count'].values[0] if not matching_claim.empty else 10
    
    horizon_days_count = len(active_days_range) if len(active_days_range) > 0 else 30
    scaled_target_claim = (target_monthly_claim / 31.0) * horizon_days_count
    
    c_base = home_bases_df[home_bases_df['captain_id'] == c]['home_base'].values[0]
    caps_pairings = pairings_by_base[c_base]
    
    total_days_worked = gp.quicksum(y[c, p] * pairings_df.loc[p, 'num_days'] for p in caps_pairings)
    observed_days_off = horizon_days_count - total_days_worked
    
    model.addConstr(dev[c] >= scaled_target_claim - observed_days_off, name=f"Fairness_SideA_{c}")
    model.addConstr(dev[c] >= observed_days_off - scaled_target_claim, name=f"Fairness_SideB_{c}")

# ==========================================
# 6. EXECUTION & OUTPUT PARSING
# ==========================================
print("\n>>> Launching Gurobi Engine Optimizer Matrix...")
model.optimize()

# Diagnostic and Extraction
if model.status == 3:  # GRB.INFEASIBLE
    print("❌ Model is Infeasible. Isolating the conflict via IIS...")
    model.computeIIS()
    model.write("model_conflict.ilp")
    print("✅ Conflict report written to 'model_conflict.ilp'. Open this file to see which rules clashed.")
elif model.status == 2:  # GRB.OPTIMAL
    print(f"\nOptimization Successful! Global Network Roster Cost: €{model.objVal:,.2f}")
    
    uncovered_count = 0
    for p in pairings_df['pairing_id']:
        if uncovered[p].X > 0.5:
            uncovered_count += 1
    print(f"Total pairings dropped to contract slack reserves (Uncovered): {uncovered_count}")

    roster_output = []
    for (c, p) in valid_y_pairs:
        if y[c, p].X > 0.5:
            p_info = pairings_df.loc[p]
            roster_output.append({
                'Captain_ID': c,
                'Home_Base': home_bases_df[home_bases_df['captain_id'] == c]['home_base'].values[0],
                'Assigned_Pairing_ID': p,
                'Start_Day': p_info['start_day'],
                'End_Day': p_info['end_day'],
                'Flying_Hours': round(p_info['flying_hours'], 2),
                'Route_Cost': round(p_info['total_cost'], 2)
            })
            
    roster_output_df = pd.DataFrame(roster_output)
    roster_output_df.to_csv("Roster_2days_ALLBASES.csv", index=False)
    
    print("\n=== ROSTER GENERATION COMPLETE! RESULTS SAVED TO 'Roster_2days_ALLBASES.csv' ===")
    
    from IPython.display import display
    display(roster_output_df.head(15))
else:
    print(f"Solver terminated with non-optimal code. Status: {model.status}")

Building pre-indexed hash maps for timeline overlap tracking...

Initializing Gurobi Engine...
Set parameter Username
Set parameter LicenseID to value 2817051
Academic license - for non-commercial use only - expires 2027-05-01
Set parameter Method to value 2
Set parameter NodefileStart to value 0.5
Instantiating 18603 base-compliant decision variables...
Instantiating uncovered pairing slack variables...
✅ Multi-objective function successfully compiled with slack penalties!
Formulating flexible pairing coverage matrices...
Formulating calendar pacing constraints...
Formulating EASA fatigue limit bounds...
Formulating absolute deviation fairness bounds...

>>> Launching Gurobi Engine Optimizer Matrix...
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i3-7100U CPU @ 2.40GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Non-default parameters:
Method  2
NodefileS

,Captain_ID,Home_Base,Assigned_Pairing_ID,Start_Day,End_Day,Flying_Hours,Route_Cost
0,640,DUS,11,1,1,0,1241.0
1,790,HAM,153,1,1,0,765.0
2,636,NUE,138,1,1,0,835.0
3,690,HAM,118,1,1,0,905.0
4,936,DUS,121,1,1,0,890.0
5,775,CGN,74,1,1,0,1008.4
6,309,DUS,27,1,1,0,1145.4
7,43,CGN,105,1,1,0,945.0
8,380,DUS,181,1,1,0,435.0
9,903,DUS,146,1,1,0,805.0


In [5]:
print(f"Total assignments generated: {len(roster_output_df)}")

Total assignments generated: 175


7 unassigned pairings are due to the fact that they are: Capacity Drops—they are feasible from a network/base perspective, but the limited pool of available, legal-to-fly captains cannot cover them without violating labor laws or pre-existing leave entitlements.

In [6]:
# Check how many pairings ended up using the 'uncovered' slack variable
uncovered_pairings = [p for p in pairings_df['pairing_id'] if uncovered[p].X > 0.5]

print("=== COVREAGE GAP DIAGNOSTIC ===")
print(f"Total pairings left uncovered: {len(uncovered_pairings)}")
if uncovered_pairings:
    print(f"Unassigned Pairing IDs: {uncovered_pairings}")
    # Show data details for those missing pairings to figure out why they failed
    display(pairings_df[pairings_df['pairing_id'].isin(uncovered_pairings)])
else:
    print("🎉 Perfection! 100% of flight pairings were fully covered by your captains.")

=== COVREAGE GAP DIAGNOSTIC ===
Total pairings left uncovered: 7
Unassigned Pairing IDs: ['15', '16', '82', '85', '87', '93', '129']


,pairing_id,base,n_legs,cost_eur,start_min,end_min,route,leg_ids,covered_legs,home_base,flying_hours,start_day,end_day,num_days,total_cost
pairing_id,,,,,,,,,,,,,,,
15,15,NUE,5,1190.4,550,2585,NUE->PMI PMI->DUS HAM->IBZ IBZ->HAM HAM->NUE,3505309478|3505309674|3505310227|3505310228|35...,[],NUE,0,1,1,1,1190.4
16,16,NUE,5,1189.2,415,2965,DUS->EDI EDI->CGN CGN->BER BER->PMI PMI->NUE,3505309629|3505309260|3505309949|3505310295|35...,[],NUE,0,1,1,1,1189.2
82,82,HAJ,4,995.0,310,2465,HAJ->PMI PMI->HAM HAM->PMI PMI->HAJ,3505309412|3505309535|3505310241|3505310194,[],HAJ,0,1,1,1,995.0
85,85,HAJ,4,985.0,360,2165,HAJ->SUF SUF->HAJ HAJ->PMI PMI->HAJ,3505309418|3505309419|3505310108|3505310196,[],HAJ,0,1,1,1,985.0
87,87,MUC,6,985.0,505,2405,MUC->HAM HAM->VIE VIE->STR STR->CTA CTA->DUS D...,3505309505|3505309550|3505309391|3505310090|35...,[],MUC,0,1,1,1,985.0
93,93,HAJ,4,970.0,1090,2875,HAJ->PMI PMI->HAJ HAJ->PMI PMI->HAJ,3505309487|3505309413|3505310195|3505310109,[],HAJ,0,1,1,1,970.0
129,129,HAJ,4,865.0,620,2525,HAJ->OLB OLB->CGN CGN->FCO FCO->HAJ,3505309416|3505310011|3505310014|3505310113,[],HAJ,0,1,1,1,865.0


In [7]:
import numpy as np

deviations = [dev[c].X for c in captains_list]
print("=== ROSTER FAIRNESS / IMBALANCE METRICS ===")
print(f"Max Imbalance Observed: {max(deviations):.2f} days")
print(f"Min Imbalance Observed: {min(deviations):.2f} days")
print(f"Average Workload Deviation: {np.mean(deviations):.2f} days")

=== ROSTER FAIRNESS / IMBALANCE METRICS ===
Max Imbalance Observed: 0.97 days
Min Imbalance Observed: 0.06 days
Average Workload Deviation: 0.50 days


In [8]:
print(f"Fairness Cost: {penalty_fairness_weight * np.sum(deviations):.2f} euros")
print(f"uncovered_penalty_cost: {PENALTY_UNCOVERED * len(uncovered_pairings):.2f} euros")

Fairness Cost: 107822.58 euros
uncovered_penalty_cost: 350000.00 euros


In [9]:
valid_captain_bases = home_bases_df['home_base'].unique().tolist()
print(f"Bases with available captains: {valid_captain_bases}")

# Count how many pairings start at bases not in that list
doomed_pairings = pairings_df[~pairings_df['home_base'].isin(valid_captain_bases)]
print(f"\nTotal pairings starting at unstaffed outstations: {len(doomed_pairings)}")


Bases with available captains: ['DUS', 'HAM', 'NUE', 'CGN', 'STR', 'BER', 'HAJ', 'MUC']

Total pairings starting at unstaffed outstations: 0
